In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.ensemble import StackingRegressor


In [13]:
train_path = "train.csv"
test_path  = "test.csv"

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

print("Train shape:", train.shape)
print("Test shape :", test.shape)

train.head()


Train shape: (15000, 10)
Test shape : (10000, 9)


,id,Sex,Length,Diameter,Height,Weight,Shucked Weight,Viscera Weight,Shell Weight,Age
0,0,F,1.6000,1.2500,0.5750,43.700754,15.025235,10.262519,13.324265,17.0
1,1,M,1.5750,1.1625,0.3750,32.417653,12.927372,6.208540,8.930093,20.0
2,2,F,1.6125,1.2750,0.4375,34.373769,14.784264,9.142714,10.772810,10.0
3,3,M,1.2250,0.9875,0.3250,18.455524,8.037083,3.926406,4.819415,9.0
4,4,I,1.3750,1.0250,0.3875,22.736299,9.879801,5.513978,6.945627,10.0


In [11]:
train_cols = set(train.columns)
test_cols  = set(test.columns)

diff = list(train_cols - test_cols)

if len(diff) == 1:
    target_col = diff[0]
else:
    target_col = train.columns[-1]

print("Detected target column:", target_col)

# Optional safety check for this task
if target_col != "age":
    print("Warning: Detected target is not 'age'. Overriding to 'age' for submission format.")
    target_col = "age"

id_col = "id"
if id_col not in test.columns:
    raise ValueError(f"ID column '{id_col}' not found in test.csv")

print("Final target column:", target_col)
print("ID column:", id_col)


Detected target column: Age
Final target column: age
ID column: id


In [14]:
train_cols = set(train.columns)
test_cols  = set(test.columns)

diff = list(train_cols - test_cols)

if len(diff) == 1:
    target_col = diff[0]
else:
    extra = list(train_cols - test_cols)
    target_col = extra[0] if len(extra) > 0 else train.columns[-1]

print("Detected target column:", target_col)

id_col = "id"
if id_col not in test.columns:
    raise ValueError(f"ID column '{id_col}' not found in test.csv")

print("ID column:", id_col)


Detected target column: Age
ID column: id


In [15]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "Valid:", X_valid.shape)


Train: (12000, 9) Valid: (3000, 9)


In [16]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)


Numeric features: 8
Categorical features: 1
